# Module 20: Smart Room Project

## Mission Briefing

Over the past 19 modules, you've learned to blink LEDs, read buttons, sense light, measure temperature, detect motion, make sounds, and more. Each module taught you ONE skill.

Today, you put it ALL together. You're going to build a **Smart Room** -- a single system that combines multiple sensors and outputs to monitor and react to its environment, just like the smart home devices people use in real life.

```
                    YOUR SMART ROOM
   ┌──────────────────────────────────────────┐
   │                                          │
   │   [Photoresistor]──> Auto Night Light    │
   │                      (LED turns on        │
   │                       when dark)          │
   │                                          │
   │   [Thermistor]────> Temperature Monitor  │
   │                      (prints warnings     │
   │                       if too hot)         │
   │                                          │
   │   [PIR Sensor]───> Motion Alarm          │
   │                      (buzzer + LED        │
   │                       on movement)        │
   │                                          │
   │   [Button]────────> System Control       │
   │                      (arm/disarm alarm)   │
   │                                          │
   │              ┌──────────┐                │
   │              │ Pico 2W  │                │
   │              │  (brain) │                │
   │              └──────────┘                │
   └──────────────────────────────────────────┘
```

This is what real engineers do -- they combine simple building blocks into complex systems. Each piece is a **subsystem** that you already know how to build. Today you connect them all together.

Let's build a smart room!

---
## Science Spotlight: System Integration and IoT

Your smart room is a tiny example of the **Internet of Things (IoT)** -- everyday objects that sense their environment and react intelligently. Smart thermostats, automatic lights, and security systems all work this way.

The key challenge isn't building each piece -- it's making them all work TOGETHER without interfering with each other. That's called **system integration**.

*Your instructor will explain how real IoT systems are designed using subsystems, thresholds, and state machines.*

---
## System Overview

Your Smart Room has **three subsystems**, each doing its own job:

| Subsystem | Sensor | Output | What It Does |
|-----------|--------|--------|-------------|
| **Auto Night Light** | Photoresistor (GP27) | LED (GP15) | LED turns on when the room gets dark |
| **Temperature Monitor** | Thermistor (GP26) | Serial print | Prints a warning if temperature is too high |
| **Motion Alarm** | PIR sensor (GP14) | Buzzer (GP13) + LED (GP15) | Buzzer sounds and LED flashes when motion is detected |

Plus a **button** (GP12) to arm/disarm the motion alarm.

```
   SENSORS (inputs)              PICO (brain)              OUTPUTS
   ┌────────────┐           ┌──────────────┐         ┌──────────┐
   │Photoresistor├──GP27──>│              │──GP15──>│   LED    │
   │ + 10k ohm  │          │              │         └──────────┘
   └────────────┘          │              │
   ┌────────────┐          │  Smart Room  │         ┌──────────┐
   │ Thermistor ├──GP26──>│   Program    │──GP13──>│  Buzzer  │
   │ + 10k ohm  │          │              │         └──────────┘
   └────────────┘          │              │
   ┌────────────┐          │              │         ┌──────────┐
   │ PIR Sensor ├──GP14──>│              │──print─>│  Serial  │
   └────────────┘          │              │         │  Monitor │
   ┌────────────┐          │              │         └──────────┘
   │  Button    ├──GP12──>│              │
   └────────────┘          └──────────────┘
```

The main loop reads ALL sensors, then checks each subsystem. This is how real embedded systems work -- a continuous loop checking everything, over and over.

---
## Wiring: The Full Smart Room Circuit

This is the most complex circuit you've built in this course! Take it step by step. Wire ONE subsystem at a time and test each before adding the next.

### Complete Circuit

```
   Photoresistor (light sensor) — Voltage Divider:
   Pico 3.3V  ──── wire ──── Photoresistor Pin 1
                              Photoresistor Pin 2 ──── junction ──── wire ──── Pico GP27 (ADC1)
                                                          |
                                                     [10k ohm]
                                                          |
                                                       Pico GND

   Thermistor (temperature sensor) — Voltage Divider:
   Pico 3.3V  ──── wire ──── Thermistor Pin 1
                              Thermistor Pin 2 ──── junction ──── wire ──── Pico GP26 (ADC0)
                                                       |
                                                  [10k ohm]
                                                       |
                                                    Pico GND

   PIR Sensor (motion detector):
   Pico VBUS (5V, pin 40) ──── wire ──── PIR VCC
   Pico GND  (pin 38)     ──── wire ──── PIR GND
   Pico GP14 (pin 19)     ──── wire ──── PIR OUT

   LED (night light + alarm indicator):
   Pico GP15 (pin 20) ──── [220 ohm] ──── LED (+) ──── LED (-) ──── GND

   Buzzer (alarm sound):
   Pico GP13 (pin 17) ──── wire ──── Buzzer (+) ──── Buzzer (-) ──── GND

   Button (arm/disarm):
   Pico GP12 (pin 16) ──── wire ──── Button ──── GND   (use PULL_UP)
```

### Step-by-Step Wiring Order

1. **Photoresistor voltage divider** -- 3.3V to photoresistor, junction to GP27, 10k ohm to GND
2. **Thermistor voltage divider** -- 3.3V to thermistor, junction to GP26, 10k ohm to GND
3. **PIR sensor** -- VBUS (5V) to VCC, GND to GND, GP14 to OUT (check your PIR pin labels!)
4. **LED** -- GP15 through 220 ohm resistor to LED to GND
5. **Buzzer** -- GP13 to buzzer (+) to buzzer (-) to GND
6. **Button** -- GP12 to one side, other side to GND

**Double-check everything before plugging in!** With this many wires, it's easy to mix up a connection.

**Important:** After plugging in USB, **wait 30-60 seconds** for the PIR sensor to warm up. Stay still during this time!

Plug in your USB cable and wait patiently.

---
## Step 1: Test Each Subsystem Individually

Before combining everything, let's make sure each piece works on its own. This is how real engineers work -- test the parts BEFORE assembling the whole.

### Test 1: Light Sensor

Does the photoresistor give you different values in light vs dark?

In [ ]:
from machine import ADC
import time

adc_light = ADC(_____)

for i in range(10):
    light_value = adc_light.read_u16()
    light_pct = int(light_value / 65535 * 100)
    print("Light:", light_pct, "%  (raw:", light_value, ")")
    time.sleep(0.5)

# Cover the sensor with your hand -- does the number drop?
# Shine a light on it -- does the number go up?

### Test 2: Temperature Sensor

Does the thermistor give you a reasonable temperature?

In [ ]:
from machine import ADC
import math
import time

adc_temp = ADC(_____)

for i in range(10):
    val = adc_temp.read_u16()
    if val == 0:
        val = 1
    resistance = 10000 * val / (65535 - val)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance / 10000))
    temp_c = temp_k - 273.15
    print("Temperature:", round(temp_c, 1), "C")
    time.sleep(0.5)

# Should read around 20-25 C at room temperature
# Hold the thermistor -- does it go up?

### Test 3: PIR Motion Sensor

Does the PIR detect motion? (Remember to wait for warm-up!)

In [ ]:
from machine import Pin
import time

pir = Pin(_____, Pin.IN)

print("Warming up PIR... stay still!")
time.sleep(3)
print("Ready! Wave your hand.")

for i in range(20):
    if pir.value() == 1:
        print("Motion detected!")
    else:
        print("All clear.")
    time.sleep(0.5)

# Wave your hand in front of the sensor
# You should see 'Motion detected!' appear

### Test 4: LED, Buzzer, and Button

Quick test to make sure all outputs and the button work.

In [ ]:
from machine import Pin, PWM
import time

led = Pin(_____, Pin.OUT)
buzzer = PWM(Pin(_____))  
buzzer.freq(1000)
buzzer.duty_u16(0)
button = Pin(_____, Pin.IN, Pin.PULL_UP)

# Test LED
print("LED on...")
led.value(1)
time.sleep(1)
led.value(0)
print("LED off.")

# Test buzzer
print("Buzzer beep...")
buzzer.duty_u16(5000)
time.sleep(0.3)
buzzer.duty_u16(0)
print("Buzzer off.")

# Test button
print("Press the button within 5 seconds...")
for i in range(50):
    if button.value() == 0:
        print("Button pressed!")
        break
    time.sleep(0.1)
else:
    print("No button press detected.")

print("All hardware tests done!")

If all four tests pass, your wiring is correct! If something doesn't work, fix it now before moving on.

**Checkpoint:** Ask your instructor to verify your hardware before continuing.

---
## Step 2: Build the Smart Room Code

Now we'll combine everything into one program. The code is organized into **functions** -- each subsystem gets its own function. The `main()` loop calls them all.

This is a real engineering pattern: break a big problem into small, testable pieces.

```
   main() loop
   ┌────────────────────────────────────────────┐
   │                                            │
   │   1. read_sensors()        Read everything │
   │          |                                 │
   │          v                                 │
   │   2. check_night_light()   Auto LED        │
   │          |                                 │
   │          v                                 │
   │   3. check_temperature()   Temp warnings   │
   │          |                                 │
   │          v                                 │
   │   4. check_motion()        Motion alarm    │
   │          |                                 │
   │          v                                 │
   │   5. sleep, repeat                         │
   │                                            │
   └────────────────────────────────────────────┘
```

Fill in the blanks to complete your Smart Room program:

In [ ]:
from machine import Pin, ADC, PWM
import math
import time

# ===== HARDWARE SETUP =====

# Sensors
adc_light = ADC(_____)           # Photoresistor on which ADC pin?
adc_temp  = ADC(_____)           # Thermistor on which ADC pin?
pir       = Pin(14, Pin.IN)      # PIR motion sensor
button    = Pin(12, Pin.IN, Pin.PULL_UP)  # Arm/disarm button

# Outputs
led       = Pin(15, Pin.OUT)     # LED for night light and alarm
buzzer    = PWM(Pin(13))         # Buzzer for alarm
buzzer.freq(2000)
buzzer.duty_u16(0)

# ===== SETTINGS (thresholds) =====

LIGHT_THRESHOLD = _____          # Below this = dark (try 30 for 30%)
TEMP_THRESHOLD  = _____          # Above this = too hot (try 30 for 30 C)

# ===== SYSTEM STATE =====

alarm_armed = False              # Is the motion alarm turned on?

# ===== FUNCTIONS =====

def read_sensors():
    """Read all sensors and return a dictionary of values."""
    # Read light level as a percentage (0-100)
    light_raw = adc_light._____()              # How do we read the ADC?
    light_pct = int(light_raw / 65535 * 100)
    
    # Read temperature in Celsius
    temp_raw = adc_temp._____()                # Same method for temperature ADC
    if temp_raw == 0:
        temp_raw = 1
    resistance = 10000 * temp_raw / (65535 - temp_raw)
    temp_k = 1 / (1/(273.15 + 25) + (1/3950) * math.log(resistance / 10000))
    temp_c = round(temp_k - 273.15, 1)
    
    # Read motion (1 = motion, 0 = no motion)
    motion = pir._____()                       # How do we read a digital pin?
    
    return {
        'light': light_pct,
        'temp': temp_c,
        'motion': motion
    }

In [ ]:
def check_night_light(light_level):
    """Turn LED on when it's dark, off when it's bright."""
    if light_level < _____:          # What variable holds our darkness threshold?
        led.value(_____)             # Dark! Turn the LED... on or off?
    else:
        led.value(_____)             # Bright! Turn the LED... on or off?

In [ ]:
def check_temperature(temp):
    """Print a warning if temperature is above the threshold."""
    if temp > _____:                 # What variable holds our temperature threshold?
        print("WARNING: Temperature is", temp, "C -- TOO HOT!")
    else:
        print("Temperature:", temp, "C -- OK")

In [ ]:
def check_motion(motion_value):
    """If alarm is armed and motion is detected, trigger the alarm."""
    global alarm_armed
    
    if alarm_armed and motion_value == _____:   # What value means motion detected?
        print("!!! MOTION ALARM !!!")
        # Flash LED and sound buzzer for 3 seconds
        for i in range(15):
            led.value(1)
            buzzer.duty_u16(_____)
            time.sleep(0.1)
            led.value(0)
            buzzer.duty_u16(_____)
            time.sleep(0.1)
        print("Alarm finished. Still armed.")

In [ ]:
def check_button():
    """Toggle the alarm armed state when button is pressed."""
    global alarm_armed
    
    if button.value() == 0:          # Button pressed (PULL_UP: 0 = pressed)
        alarm_armed = not alarm_armed
        if alarm_armed:
            print(">> ALARM ARMED <<")
            # Two short beeps to confirm
            for i in range(2):
                buzzer.duty_u16(3000)
                time.sleep(0.1)
                buzzer.duty_u16(0)
                time.sleep(0.1)
        else:
            print(">> ALARM DISARMED <<")
            # One long beep to confirm
            buzzer.duty_u16(3000)
            time.sleep(0.3)
            buzzer.duty_u16(0)
        time.sleep(0.3)              # Debounce

---
## Step 3: The Main Loop

Now we put it all together! The `main()` function runs an infinite loop that:
1. Reads all sensors
2. Checks the button
3. Runs each subsystem
4. Prints a status report
5. Waits briefly, then repeats

Fill in the blanks:

In [ ]:
def main():
    """Main loop: read sensors, check subsystems, repeat."""
    print("=============================")
    print("   SMART ROOM STARTING UP")
    print("=============================")
    print("Waiting for PIR sensor to warm up...")
    time.sleep(5)
    print("System ready!")
    print("Press the button to arm/disarm the motion alarm.")
    print()
    
    loop_count = 0
    
    while True:
        # Check if button was pressed
        check_button()
        
        # Read all sensors
        data = _____()                   # Which function reads all sensors?
        
        # Run each subsystem
        check_night_light(data['light'])
        check_temperature(data['_____'])  # Which key in the dictionary has temperature?
        check_motion(data['motion'])
        
        # Print status every 10 loops (~5 seconds)
        loop_count += 1
        if loop_count % 10 == 0:
            armed_status = "ARMED" if alarm_armed else "disarmed"
            print("--- Status: Light", data['light'], "% | Temp",
                  data['temp'], "C | Alarm:", armed_status, "---")
        
        time.sleep(0.5)

# Run the smart room!
main()

If everything is working, you should see:
- Temperature readings printing continuously
- LED turning on when you cover the photoresistor (simulating darkness)
- After pressing the button to arm: buzzer sounds when the PIR detects motion
- Status reports every ~5 seconds

**Troubleshooting:**
- If the LED doesn't respond to light changes, check your photoresistor wiring and try adjusting `LIGHT_THRESHOLD`
- If temperature seems wrong, make sure the thermistor is on GP26 and the photoresistor on GP27 (not swapped!)
- If motion alarm never triggers, make sure you pressed the button to arm it first

---
## Experiments

Try these with your Smart Room running:

1. **Threshold tuning:** Change `LIGHT_THRESHOLD` to make the night light more or less sensitive. What value works best for your room?

2. **Temperature alert:** Hold the thermistor between your fingers until it triggers the temperature warning. What threshold did you set?

3. **Alarm test:** Arm the system, then walk past the PIR sensor. Does the alarm trigger? Try sneaking past slowly -- can you fool it?

4. **Combined test:** Cover the photoresistor (night light should turn on), then wave past the PIR (alarm should trigger if armed). Both subsystems should work at the same time!

5. **Timing:** How fast does the main loop run? Add `print(time.ticks_ms())` to see. Why might a faster or slower loop matter?

---
## Challenge: Add Your Own Feature!

Your Smart Room works -- but can you make it smarter? Pick one (or more!) of these ideas and add it to your system:

### Idea 1: Temperature + Night Light Combo
If it's dark AND hot, flash the LED as a visual alert (since you might be sleeping and can't see the serial monitor).

### Idea 2: Entry Countdown
When motion is detected, give a 5-second warning countdown (beep... beep... beep...) before the full alarm. This gives the homeowner time to press the button to disarm.

### Idea 3: Status Dashboard
If you have an OLED display (Module 12), show all sensor readings on the screen instead of printing to serial.

### Idea 4: Web Dashboard (Advanced)
If you completed Modules 15-16, add a web server that shows sensor data in a browser! Use your `connect_wifi()` and web server code from those modules.

### Idea 5: Your Own Idea!
What feature would make YOUR room smarter? Add it!

```
   My custom feature: _______________________________________________
   
   What it does: ____________________________________________________
   
   Components needed: _______________________________________________
```

In [ ]:
# Your challenge code here!


---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Integration** | Combining separate parts into one working system |
| **Subsystem** | A smaller, self-contained part of a bigger system (e.g., the night light is one subsystem) |
| **Threshold** | A boundary value that triggers an action (e.g., if light drops below 30%, turn on LED) |
| **State machine** | A system that behaves differently depending on what mode it's in (e.g., armed vs disarmed) |
| **IoT (Internet of Things)** | Everyday objects connected to sensors and networks that react to their environment |

---
*Circuit Explorers -- Module 20: Smart Room Project*